# Project 2 — Local Markdown Knowledge Bot
## Query Your Markdown Notes with LlamaIndex + Ollama

**What you'll learn:**
- Parse markdown files into structured documents
- Build a LlamaIndex VectorStoreIndex over your notes
- Query your knowledge base conversationally
- Handle metadata (titles, tags, dates) for better retrieval

**Stack:** Ollama · LlamaIndex · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Generate Sample Markdown Notes

In [ ]:
from pathlib import Path

notes_dir = Path("sample_notes"); notes_dir.mkdir(exist_ok=True)

notes = {
    "python_tips.md": """# Python Tips
## List Comprehensions
List comprehensions are a concise way to create lists:
`squares = [x**2 for x in range(10)]`

## Context Managers
Use `with` statements for resource management. They ensure cleanup happens
even if exceptions occur. Common for file handling, database connections.

## Type Hints
Type hints improve code readability: `def greet(name: str) -> str:`
""",
    "git_workflow.md": """# Git Workflow Guide
## Branching Strategy
- `main` — production-ready code
- `develop` — integration branch
- `feature/*` — new features
- `hotfix/*` — urgent production fixes

## Common Commands
- `git rebase -i HEAD~3` — interactive rebase last 3 commits
- `git stash` — temporarily save uncommitted changes
- `git cherry-pick <hash>` — apply specific commit to current branch
""",
    "docker_basics.md": """# Docker Basics
## Key Concepts
- **Image**: Read-only template with instructions for creating a container
- **Container**: Runnable instance of an image
- **Dockerfile**: Script that defines how to build an image
- **Volume**: Persistent data storage that survives container restarts

## Essential Commands
```bash
docker build -t myapp .
docker run -p 8080:80 myapp
docker-compose up -d
```
""",
}
for fname, content in notes.items():
    (notes_dir / fname).write_text(content, encoding="utf-8")
print(f"Created {len(notes)} sample markdown notes in {notes_dir}/")

## Step 2 — Configure LlamaIndex with Local Ollama

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("LlamaIndex configured with local Ollama models.")

## Step 3 — Load Documents and Build Index

In [ ]:
documents = SimpleDirectoryReader("sample_notes").load_data()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

index = VectorStoreIndex.from_documents(documents, show_progress=True)
print("\nVector index built successfully!")

## Step 4 — Query Your Notes

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

queries = [
    "How do I use list comprehensions in Python?",
    "What is the Git branching strategy?",
    "How do I build a Docker image?",
    "What are context managers used for?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    response = query_engine.query(q)
    print(f"A: {response}")
    print(f"Sources: {[n.metadata.get('file_name','?') for n in response.source_nodes]}")

## Step 5 — Chat Mode with Memory

In [ ]:
from llama_index.core.chat_engine import CondenseQuestionChatEngine

chat_engine = CondenseQuestionChatEngine.from_defaults(query_engine=query_engine)

conversation = [
    "Tell me about Docker volumes",
    "How do I persist data with them?",
    "What command starts containers in the background?",
]
for msg in conversation:
    print(f"\nUser: {msg}")
    response = chat_engine.chat(msg)
    print(f"Bot: {response}")

## Step 6 — Add Metadata Filtering

In [ ]:
from llama_index.core import Document

# Rebuild with explicit metadata
enriched_docs = []
for doc in documents:
    fname = doc.metadata.get('file_name', '')
    topic = fname.replace('.md', '').replace('_', ' ').title()
    enriched_docs.append(Document(
        text=doc.text,
        metadata={"topic": topic, "file": fname, "type": "notes"}
    ))

enriched_index = VectorStoreIndex.from_documents(enriched_docs)
enriched_qe = enriched_index.as_query_engine(similarity_top_k=2)

resp = enriched_qe.query("What are essential Docker commands?")
print(f"Answer: {resp}")
print(f"Metadata: {[n.metadata for n in resp.source_nodes]}")

## What You Learned
- **Markdown loading** with SimpleDirectoryReader
- **LlamaIndex VectorStoreIndex** for semantic search
- **Chat engine** with conversation memory
- **Metadata enrichment** for better filtering